# Week 6 Automated Reporting Pipeline

## Objective

Build an automated reporting pipeline for Aido Rover telemetry data.

The pipeline includes:

1. Data ingestion and validation
2. Health KPI calculation
3. Anomaly detection summary
4. Summary table generation
5. Chart generation
6. Markdown report generation

In [8]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

In [9]:
def ingest(path):

    df = pd.read_csv(path)

    print("Dataset shape:")
    print(df.shape)


    print("\nMissing values:")
    print(df.isnull().sum())


    numeric_columns = df.select_dtypes(
        include="number"
    ).columns


    df[numeric_columns] = (
        df[numeric_columns]
        .interpolate(
            limit_direction="both"
        )
    )


    print("\nData ingestion completed.")

    return df

In [10]:
rover_df = ingest(
    "../data/synthetic_rover_data.csv"
)

rover_df.head()

Dataset shape:
(12000, 11)

Missing values:
timestamp           0
latitude          360
longitude         360
lidar_distance      0
battery_soc         0
wheel_torque_1      0
wheel_torque_2      0
wheel_torque_3      0
wheel_torque_4      0
temperature         0
status              0
dtype: int64

Data ingestion completed.


,timestamp,latitude,longitude,lidar_distance,battery_soc,wheel_torque_1,wheel_torque_2,wheel_torque_3,wheel_torque_4,temperature,status
0,2026-07-01 00:00:00,43.072474,-89.400668,6.214978,101.499580,30.498199,27.707335,27.063768,29.265164,26.144969,Normal
1,2026-07-01 00:01:00,43.073330,-89.401182,2.904787,101.481014,31.638698,30.454532,24.826910,30.484939,27.499129,Fault
2,2026-07-01 00:02:00,43.073878,-89.401202,30.000000,98.925882,34.718033,29.222000,31.708100,32.937053,18.317683,Normal
3,2026-07-01 00:03:00,43.072368,-89.402706,5.984263,100.572100,36.340124,17.562884,36.368883,29.765023,23.060746,Normal
4,2026-07-01 00:04:00,43.073881,-89.402177,6.773406,102.499250,30.074116,27.359921,33.890405,34.695564,23.604358,Normal


In [11]:
def create_features(df):

    WINDOW = 20


    sensor_columns = [
        "battery_soc",
        "lidar_distance",
        "wheel_torque_1",
        "wheel_torque_2",
        "wheel_torque_3",
        "wheel_torque_4",
        "temperature"
    ]


    anomaly_features = []


    for column in sensor_columns:


        rolling_mean = (
            df[column]
            .rolling(WINDOW)
            .mean()
        )


        rolling_std = (
            df[column]
            .rolling(WINDOW)
            .std()
        )


        feature_name = (
            f"{column}_variability_score"
        )


        df[feature_name] = (
            rolling_std /
            rolling_mean.abs()
            .replace(0,np.nan)
        )


        anomaly_features.append(
            feature_name
        )


    df["composite_anomaly_score"] = (
        df[anomaly_features]
        .mean(axis=1)
    )


    print(
        "Feature engineering completed."
    )


    return df

In [12]:
rover_df = create_features(
    rover_df
)

Feature engineering completed.


In [13]:
rover_df[
    [
        "temperature_variability_score",
        "composite_anomaly_score"
    ]
].head(25)

,temperature_variability_score,composite_anomaly_score
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
5,NaN,NaN
6,NaN,NaN
7,NaN,NaN
8,NaN,NaN
9,NaN,NaN


In [14]:
def compute_health_kpis(df):


    kpis = {


        "mean_battery_soc":
        df["battery_soc"].mean(),



        "fault_rate":
        (
            df["status"]
            .eq("fault")
            .mean()
        ),



        "mean_lidar_distance":
        df["lidar_distance"].mean(),



        "wheel_torque_imbalance":
        (
            df[
                [
                "wheel_torque_1",
                "wheel_torque_2",
                "wheel_torque_3",
                "wheel_torque_4"
                ]
            ]
            .std(axis=1)
            .mean()
        )

    }


    return pd.DataFrame(
        [kpis]
    )

In [15]:
kpi_table = compute_health_kpis(
    rover_df
)

kpi_table

,mean_battery_soc,fault_rate,mean_lidar_distance,wheel_torque_imbalance
0,59.987492,0.0,8.445379,4.89065


In [16]:
def compute_anomaly_summary(df):


    threshold = (
        df["composite_anomaly_score"]
        .quantile(0.95)
    )


    high_anomaly = df[
        df["composite_anomaly_score"]
        >= threshold
    ]


    result = {


        "anomaly_threshold":
        threshold,


        "high_risk_count":
        len(high_anomaly),


        "max_anomaly_score":
        high_anomaly[
            "composite_anomaly_score"
        ]
        .max()

    }


    return pd.DataFrame(
        [result]
    )

In [17]:
anomaly_table = compute_anomaly_summary(
    rover_df
)

anomaly_table

,anomaly_threshold,high_risk_count,max_anomaly_score
0,0.348934,600,0.403913


In [19]:
def generate_summary_table(
    kpi,
    anomaly
):

    return pd.concat(
        [
            kpi,
            anomaly
        ],
        axis=1
    )

In [20]:
summary_table = generate_summary_table(
    kpi_table,
    anomaly_table
)

summary_table

,mean_battery_soc,fault_rate,mean_lidar_distance,wheel_torque_imbalance,anomaly_threshold,high_risk_count,max_anomaly_score
0,59.987492,0.0,8.445379,4.89065,0.348934,600,0.403913


In [21]:
output_dir = Path(
    "../dashboard/data"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [22]:
def generate_charts(
    df,
    output_dir
):


    # 1 Battery trend

    plt.figure(figsize=(10,4))

    plt.plot(
        df["battery_soc"]
    )

    plt.title(
        "Battery SoC Trend"
    )

    plt.xlabel(
        "Time Step"
    )

    plt.ylabel(
        "Battery SoC"
    )

    plt.savefig(
        output_dir /
        "battery_soc_trend.png",
        bbox_inches="tight"
    )

    plt.close()



    # 2 Fault rate


    fault_rate = (
        df["status"]
        .eq("fault")
        .rolling(100)
        .mean()
    )


    plt.figure(figsize=(10,4))

    plt.plot(
        fault_rate
    )


    plt.title(
        "Fault Rate Over Time"
    )


    plt.savefig(
        output_dir /
        "fault_rate.png",
        bbox_inches="tight"
    )

    plt.close()



    # 3 Top anomaly


    top_anomaly = (
        df[
            [
            "timestamp",
            "composite_anomaly_score"
            ]
        ]
        .sort_values(
            "composite_anomaly_score",
            ascending=False
        )
        .head(20)
    )


    plt.figure(figsize=(10,4))


    plt.bar(
        range(
            len(top_anomaly)
        ),
        top_anomaly[
            "composite_anomaly_score"
        ]
    )


    plt.title(
        "Top Anomalous Timesteps"
    )


    plt.savefig(
        output_dir /
        "top_anomalies.png",
        bbox_inches="tight"
    )


    plt.close()



    print(
        "Charts generated."
    )

In [23]:
generate_charts(
    rover_df,
    output_dir
)

Charts generated.


In [24]:
def generate_report(
    summary,
    path
):


    text = f"""
# Aido Rover Health Report


## KPI Summary


{summary.to_markdown(index=False)}


## Findings


The automated reporting pipeline calculated
fleet health indicators including battery
performance, fault rate, LiDAR condition,
and anomaly detection.


## Recommendation


High anomaly periods should be reviewed
for possible maintenance actions.
"""


    with open(
        path,
        "w"
    ) as file:

        file.write(text)



    print(
        "Report generated."
    )

In [26]:
generate_report(
    summary_table,
    "../reports/W06_Aido_Rover_Report.md"
)

Report generated.


In [27]:
print("Starting Week 6 Pipeline...\n")


rover_df = ingest(
    "../data/synthetic_rover_data.csv"
)


rover_df = create_features(
    rover_df
)


kpi_table = compute_health_kpis(
    rover_df
)


anomaly_table = compute_anomaly_summary(
    rover_df
)


summary_table = generate_summary_table(
    kpi_table,
    anomaly_table
)


generate_charts(
    rover_df,
    output_dir
)


generate_report(
    summary_table,
    "../reports/W06_Aido_Rover_Report.md"
)


print(
    "\nWeek 6 Pipeline Completed Successfully!"
)

Starting Week 6 Pipeline...

Dataset shape:
(12000, 11)

Missing values:
timestamp           0
latitude          360
longitude         360
lidar_distance      0
battery_soc         0
wheel_torque_1      0
wheel_torque_2      0
wheel_torque_3      0
wheel_torque_4      0
temperature         0
status              0
dtype: int64

Data ingestion completed.
Feature engineering completed.
Charts generated.
Report generated.

Week 6 Pipeline Completed Successfully!
